In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from medmnist import PathMNIST

# 1. Préparation des données (avec Resize 224)

In [13]:
# On récupère les poids officiels de ResNet-18
weights = models.ResNet18_Weights.IMAGENET1K_V1
transform_resnet = weights.transforms()

print("Chargement des données avec les transformations officielles d'ImageNet")
train_dataset_res = PathMNIST(split='train', transform=transform_resnet, download=True)
test_dataset_res = PathMNIST(split='test', transform=transform_resnet, download=True)

train_loader_res = DataLoader(train_dataset_res, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader_res = DataLoader(test_dataset_res, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 10

Chargement des données avec les transformations officielles d'ImageNet


# EXPERIMENT A : FINE-TUNE ONLY THE HEAD (FROZEN)

In [14]:
model_frozen = models.resnet18(weights=weights)

# On gèle TOUTES les couches
for param in model_frozen.parameters():
    param.requires_grad = False

# On remplace la dernière couche (qui sera dégelée par défaut)
num_ftrs = model_frozen.fc.in_features
model_frozen.fc = nn.Linear(num_ftrs, 9)
model_frozen = model_frozen.to(device)

# On donne à l'optimiseur UNIQUEMENT les paramètres de la dernière couche
optimizer_frozen = optim.Adam(model_frozen.fc.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):
    model_frozen.train()
    for images, labels in train_loader_res:
        images, labels = images.to(device), labels.to(device).squeeze()
        optimizer_frozen.zero_grad()
        outputs = model_frozen(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_frozen.step()
    print(f"Frozen - Epoch {epoch+1}/{epochs} terminée.")

# Test
model_frozen.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader_res:
        images, labels = images.to(device), labels.to(device).squeeze()
        outputs = model_frozen(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
acc_frozen = (correct / total) * 100
print(f"Précision TEST (Experiment A - Frozen) : {acc_frozen:.2f}%")

Frozen - Epoch 1/10 terminée.
Frozen - Epoch 2/10 terminée.
Frozen - Epoch 3/10 terminée.
Frozen - Epoch 4/10 terminée.
Frozen - Epoch 5/10 terminée.
Frozen - Epoch 6/10 terminée.
Frozen - Epoch 7/10 terminée.
Frozen - Epoch 8/10 terminée.
Frozen - Epoch 9/10 terminée.
Frozen - Epoch 10/10 terminée.
Précision TEST (Experiment A - Frozen) : 85.79%


# EXPERIMENT B : FULL FINE-TUNING (UNFROZEN)

In [15]:
model_full = models.resnet18(weights=weights)

# On remplace la dernière couche (et on ne gèle rien, tout reste entraînable)
model_full.fc = nn.Linear(num_ftrs, 9)
model_full = model_full.to(device)

# On donne à l'optimiseur TOUS les paramètres
# On utilise un learning rate plus petit (0.0001) pour ne pas détruire les poids pré-entraînés
optimizer_full = optim.Adam(model_full.parameters(), lr=0.0001)

for epoch in range(epochs):
    model_full.train()
    for images, labels in train_loader_res:
        images, labels = images.to(device), labels.to(device).squeeze()
        optimizer_full.zero_grad()
        outputs = model_full(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_full.step()
    print(f"Full Fine-Tuning - Epoch {epoch+1}/{epochs} terminée.")

# Test
model_full.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader_res:
        images, labels = images.to(device), labels.to(device).squeeze()
        outputs = model_full(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
acc_full = (correct / total) * 100
print(f"Précision TEST (Experiment B - Full) : {acc_full:.2f}%")

Full Fine-Tuning - Epoch 1/10 terminée.
Full Fine-Tuning - Epoch 2/10 terminée.
Full Fine-Tuning - Epoch 3/10 terminée.
Full Fine-Tuning - Epoch 4/10 terminée.
Full Fine-Tuning - Epoch 5/10 terminée.
Full Fine-Tuning - Epoch 6/10 terminée.
Full Fine-Tuning - Epoch 7/10 terminée.
Full Fine-Tuning - Epoch 8/10 terminée.
Full Fine-Tuning - Epoch 9/10 terminée.
Full Fine-Tuning - Epoch 10/10 terminée.
Précision TEST (Experiment B - Full) : 86.02%


### 4.1 — Résultats des expériences de Transfer Learning

Voici les précisions exactes sur l'ensemble de test pour nos deux expériences (l'objectif de $\ge$ 85% est atteint dans les deux cas) :

* **Expérience A (Couches gelées - Frozen) :** 85.79%
* **Expérience B (Fine-tuning complet - Full) :** 86.02%

**Comparaison :**
L'expérience de Fine-tuning complet (Expérience B) a obtenu les meilleurs résultats, surpassant le modèle gelé de **0.23 point de pourcentage**. Bien que l'écart soit relativement faible sur un petit nombre d'époques, cela démontre que permettre au réseau d'adapter la totalité de ses poids (et pas seulement la tête de classification) aux textures spécifiques des tissus histologiques permet d'optimiser les performances finales.

---

### 4.2 — Conséquences de l'agrandissement (Upscaling 28x28 vers 224x224)

**Conséquence négative de cet agrandissement extrême :**
Un agrandissement x8 (upscaling) n'ajoute absolument aucune véritable information biologique à l'image. Au contraire, il introduit des **artefacts d'interpolation** (flou, lissage artificiel ou pixellisation). Le modèle "voit" donc de faux dégradés et des bords de cellules adoucis artificiellement au lieu des structures nettes d'origine. De plus, cela fait exploser le coût en mémoire et en temps de calcul de manière totalement artificielle.

**Stratégie alternative :**
Au lieu de modifier l'image pour l'adapter au modèle, une meilleure stratégie consiste à **modifier le modèle pour l'adapter à l'image**. Pour un ResNet-18, on peut modifier sa première couche de convolution (`conv1`) pour qu'elle ait un `stride=1` et un `kernel_size=3` (au lieu de 7), et **supprimer la couche de Max Pooling** initiale. Ainsi, le modèle peut traiter nativement des petites images de 28x28 sans détruire l'information spatiale dès le début du réseau.

---

### 4.3 — Pourquoi le Transfer Learning fonctionne-t-il ici ?

**L'affirmation du camarade est-elle correcte ?**
**Non, cette affirmation est fausse.** ImageNet est une base de données de photographies naturelles (chiens, chats, voitures, paysages...) et ne contient pas de base massive d'images médicales ou d'histologie. 

**La véritable raison du succès du Transfer Learning :**
Si le transfert d'apprentissage fonctionne si bien, c'est grâce à la nature hiérarchique des réseaux de neurones convolutifs (CNN). Les **premières couches** d'un ResNet pré-entraîné n'ont pas appris à reconnaître un concept complexe comme un "chien", mais ont appris à extraire des **caractéristiques visuelles universelles de bas niveau** (low-level features) : détection de contours, de gradients, de lignes, d'angles et de taches de couleurs (blobs). 

Ces blocs de construction visuels fondamentaux sont tout aussi valables et nécessaires pour analyser les membranes cellulaires ou les noyaux d'une image médicale que pour délimiter les contours d'une voiture. Le Transfer Learning nous évite de devoir réapprendre à l'ordinateur "comment voir" les formes de base ; il n'a plus qu'à apprendre à les combiner dans ses dernières couches pour diagnostiquer des tissus.